In [2]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../data/raw/airtel.db')  
 
def q(sql):                                     

    return pd.read_sql(sql, con)

In [3]:
def run_sql_file(path):
    with open(path, 'r') as f:
        script = f.read()
    con.executescript(script)
    con.commit()
    print('ran',path)

In [4]:
run_sql_file('../sql/01_revenue_customer.sql')

ran ../sql/01_revenue_customer.sql


In [5]:
q("""
    SELECT COUNT(*) AS total_customers,
    ROUND(AVG(arpu_inr), 2) AS average_recorded_arpu,
    COUNT(DISTINCT arpu_inr) AS distinct_arpu_values
FROM customers;
""")

,total_customers,average_recorded_arpu,distinct_arpu_values
0,22000,421.65,13


In [6]:
q("""
SELECT 
    COUNT(*) AS customers_without_payments
FROM customers AS c
WHERE NOT EXISTS (
    SELECT 1
    FROM recharges AS r
    WHERE r.customer_id = c.customer_id
      AND r.recharge_date >= '2025-01-01'
      AND r.recharge_date < '2025-07-01'
);
""")

,customers_without_payments
0,3711


In [7]:
q("""SELECT name, type
FROM sqlite_master
WHERE name = 'v_revenue_customer';
""")

,name,type
0,v_revenue_customer,view


In [8]:
run_sql_file('../sql/02_usage_customer.sql')

ran ../sql/02_usage_customer.sql


In [9]:
q("""SELECT 
    SUM(CASE WHEN recharge_count > 0 THEN 1 ELSE 0 END)
        AS customers_with_payments,
    SUM(CASE WHEN recharge_count > 0 THEN 1 ELSE 0 END)
            AS customers_with_payments
FROM v_revenue_customer;""")

,customers_with_payments,customers_with_payments
0,18289,18289


In [10]:
run_sql_file("../sql/03_complaints_customer.sql")

ran ../sql/03_complaints_customer.sql


In [11]:
q("""select count(*) from v_complaint_customer;""" )

,count(*)
0,22000


In [12]:
q("""select * from v_complaint_customer;""" )

,customer_id,complaint_count,unresolved_count,resolved_count,avg_resolution_days
0,1,1,1,0,NaN
1,2,1,1,0,NaN
2,3,0,0,0,NaN
3,4,0,0,0,NaN
4,5,2,1,1,13.0
...,...,...,...,...,...
21995,21996,2,1,1,11.0
21996,21997,1,0,1,13.0
21997,21998,1,0,1,10.0
21998,21999,0,0,0,NaN


In [13]:
q("""select count(*) from v_complaint_customer where complaint_count = 0;""")

,count(*)
0,6631


In [14]:
run_sql_file("../sql/04_plan_clean.sql")

ran ../sql/04_plan_clean.sql


In [15]:
q("""select * from v_plans_clean;""")

,plan_id,plan_name,plan_type,price_inr,validity_days,data_quota,quota_basis,quota_gb,monthly_gb,ott_benefit,is_5g
0,1,Prepaid Lite 99,Prepaid,99,14,0.5 GB/day,per day,0.5,15.0,NaN,0
1,2,Prepaid Smart 155,Prepaid,155,24,1 GB/day,per day,1.0,30.0,NaN,0
2,3,Prepaid Value 209,Prepaid,209,28,1 GB/day,per day,1.0,30.0,NaN,0
3,4,Prepaid Popular 299,Prepaid,299,28,1.5 GB/day,per day,1.5,45.0,NaN,1
4,5,Prepaid Data 349,Prepaid,349,28,2.5 GB/day,per day,2.5,75.0,NaN,1
5,6,Prepaid Entertainment 399,Prepaid,399,28,2.5 GB/day,per day,2.5,75.0,OTT-Lite,1
6,7,Prepaid Max 479,Prepaid,479,56,1.5 GB/day,per day,1.5,45.0,NaN,1
7,8,Prepaid Quarterly 719,Prepaid,719,84,2 GB/day,per day,2.0,60.0,NaN,1
8,9,Prepaid Annual 1799,Prepaid,1799,365,2 GB/day,per day,2.0,60.0,OTT,1
9,10,Postpaid 399,Postpaid,399,30,40 GB/month,per month,40.0,40.0,OTT-Lite,1


In [16]:
run_sql_file("../sql/05_usage_month.sql")

ran ../sql/05_usage_month.sql


In [17]:
run_sql_file("../sql/06_customer_features.sql""")

ran ../sql/06_customer_features.sql


In [18]:
q("""select * from customer_features""")

,customer_id,activation_date,tenure_days,tenure_years,observed_days,circle_name,region,plan_name,plan_type,price_inr,...,actual_arpu_6m,stated_arpu,arpu_gap,complaint_count,unresolved_count,resolved_count,avg_resolution_days,r_score,f_score,m_score
0,9,2022-09-04,1030,2,180,Delhi,North,Prepaid Data 349,Prepaid,349,...,290.83,374,83.17,3,0,3,9.33,2,3,3
1,31,2024-02-10,506,1,180,Delhi,North,Prepaid Quarterly 719,Prepaid,719,...,239.67,257,17.33,0,0,0,NaN,3,2,3
2,151,2022-07-22,1074,2,180,Delhi,North,Prepaid Data 349,Prepaid,349,...,349.00,374,25.00,4,1,3,5.33,2,3,3
3,186,2022-01-04,1273,3,180,Delhi,North,Prepaid Quarterly 719,Prepaid,719,...,239.67,257,17.33,0,0,0,NaN,4,2,3
4,207,2024-12-14,198,0,91,Delhi,North,Prepaid Value 209,Prepaid,209,...,69.67,224,154.33,3,0,3,5.33,1,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21995,21805,2022-09-03,1031,2,180,North East,East,Prepaid Max 479,Prepaid,479,...,159.67,257,97.33,2,0,2,10.00,2,2,2
21996,21831,2022-12-18,925,2,180,North East,East,Prepaid Quarterly 719,Prepaid,719,...,239.67,257,17.33,0,0,0,NaN,2,2,3
21997,21906,2024-05-16,410,1,180,North East,East,Prepaid Entertainment 399,Prepaid,399,...,332.50,428,95.50,0,0,0,NaN,2,3,3
21998,21917,2024-02-25,491,1,180,North East,East,Prepaid Data 349,Prepaid,349,...,290.83,374,83.17,1,0,1,1.00,4,3,3


In [19]:
q("""SELECT COUNT(*)                                     AS rows,
       COUNT(DISTINCT customer_id)                  AS subs,
       SUM(usage_days = 0)                          AS no_usage,
       SUM(recharge_count = 0)                      AS no_payment,
       SUM(complaint_count = 0)                     AS never_complained,
       ROUND(AVG(tenure_days), 1)                   AS tenure,
       ROUND(AVG(actual_arpu_6m), 2)                AS actual_arpu,
       ROUND(100.0 * AVG(is_churned), 2)            AS churn
FROM customer_features;""")

,rows,subs,no_usage,no_payment,never_complained,tenure,actual_arpu,churn
0,22000,22000,4,3711,6631,697.6,344.46,23.75


In [20]:
q("""SELECT c.customer_id,                      -- one row per sub
         NTILE(4) OVER (ORDER BY             -- 1 = longest
           COALESCE(u.recency_days, 999)     -- silence
           DESC) AS r_score,                 -- the R score
         NTILE(4) OVER (ORDER BY             -- 4 = pays most
           r.recharge_count) AS f_score,     -- often
         NTILE(4) OVER (ORDER BY             -- 4 = pays the
           r.recharge_total_6m) AS m_score   -- most money
  FROM   customers c                         -- all 22,000
  JOIN   v_usage_customer   u ON u.customer_id = c.customer_id -- Fix 2
  JOIN   v_revenue_customer r ON r.customer_id = c.customer_id ;-- Fix 1""")

,customer_id,r_score,f_score,m_score
0,266,1,1,1
1,10684,1,1,1
2,13508,1,3,3
3,19111,1,3,3
4,11341,1,1,1
...,...,...,...,...
21995,18304,4,4,3
21996,19504,4,4,3
21997,19685,4,4,3
21998,21196,4,4,3
